# ⚡ CircuitAI — Colab GPU Backend
**Step 1**: Runtime → Change runtime type → T4 or L4 GPU

**Step 2**: Run cells in order.

**Step 3**: Wait for `✅ All models ready`, then paste the `trycloudflare.com` URL into the frontend.

In [ ]:
# ── Cell 0: Mount Drive & Set Working Directory ─────────────────
import sys, os
from google.colab import drive

drive.mount('/content/drive')

# ⚠️ Update this path if your folder is in a different location
BACKEND_DIR = '/content/drive/MyDrive/knowledge_graph/backend'

os.chdir(BACKEND_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir()}')

In [ ]:
# ── Cell 1: Install Cloudflared ────────────────────────────────
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb
print('Cloudflared installed ✓')

In [ ]:
# ── Cell 2: Install Python dependencies ────────────────────────
!pip install -r requirements.txt -q
print('Python deps installed ✓')

In [ ]:
# ── Cell 3: Start FastAPI server + Cloudflare Tunnel ───────────
#
# SAFE RE-RUN DESIGN:
#   Instead of killing OS PIDs (which would kill the Jupyter kernel itself
#   when uvicorn runs as a thread), we keep a reference to the uvicorn
#   Server object and call server.should_exit = True to stop it gracefully.
# ────────────────────────────────────────────────────────────────

import subprocess, threading, time, uvicorn
from server_utils import wait_for_server, acquire_lock, release_lock, find_free_port, is_port_in_use
from main import app

PORT = 8000

# ── Step 1: Stop any existing uvicorn server from a previous run
# We persist the server object in a global so re-running this cell
# can gracefully stop the old one instead of killing PIDs.
try:
    if _circuitai_server and not _circuitai_server.should_exit:
        print('[startup] Stopping previous server instance...')
        _circuitai_server.should_exit = True
        time.sleep(3)   # give uvicorn time to release the port
        print('[startup] Previous server stopped ✓')
except NameError:
    pass  # first run — no previous server

# ── Step 2: Find a free port (fallback if 8000 still busy)
if is_port_in_use(PORT):
    PORT = find_free_port(PORT + 1)
    print(f'[startup] Port 8000 still busy → using port {PORT}')
else:
    print(f'[startup] Port {PORT} is free ✓')

# ── Step 3: Build and start the uvicorn server
config = uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='info')
_circuitai_server = uvicorn.Server(config)   # global — survives cell re-runs

acquire_lock()

def _run_server():
    _circuitai_server.run()
    release_lock()
    print('[shutdown] Server stopped cleanly ✓')

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
print(f'[startup] Server thread started on port {PORT}')

# ── Step 4: Wait for /health (models finish loading inside lifespan)
ok = wait_for_server(PORT, timeout=150)
if not ok:
    raise RuntimeError('Server did not start in time — check logs above for errors.')

# ── Step 5: Start Cloudflare tunnel AFTER server is confirmed healthy
print('\nStarting Cloudflare tunnel...')
p = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in p.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        tokens = line.split()
        public_url = next((t for t in tokens if 'trycloudflare.com' in t), None)
        if public_url:
            print('\n' + '='*60)
            print(f'  🔥 PUBLIC URL : {public_url}')
            print(f'  Backend port : {PORT}')
            print('  Paste this URL into CircuitAI frontend > Connection field')
            print('='*60)
        break